# Unity Catalog — Governance, Lineage, Access Control on Databricks

This notebook demonstrates a **requests-based** approach to Unity Catalog governance on Databricks using the workspace REST APIs.

## Mental model

Unity Catalog organizes governed data into a **three-level namespace**:

- **Catalog** → top-level governance boundary
- **Schema** → logical grouping inside a catalog
- **Table / View / Volume** → actual governed asset

For a Citi-style telemetry platform, this gives one place to reason about:

- endpoint metadata and business ownership
- metrics and alert datasets
- data classification and tags
- lineage between upstream and downstream tables
- access control at catalog, schema, and table level

The examples below are grounded in the supplied learning stack and Citi telemetry narrative. fileciteturn1file0


In [ ]:

from __future__ import annotations

import json
import os
import sys
import textwrap
from dataclasses import dataclass
from datetime import datetime, timezone
from typing import Any, Dict, Iterable, List, Optional

import requests
from requests import Response, Session

HOST = "https://dbc-9f35a83d-b4e7.cloud.databricks.com"
SQL_WAREHOUSE_ID = "b6657f31d1e7a179"

POSTGRES = {
    "host": "localhost",
    "port": 5432,
    "database": "de_telemetry",
    "user": "de_admin",
    "password": "DeAdmin2026!",
}

DATASET_CONTEXT = {
    "endpoints": {
        "rows": 10_000,
        "primary_key": "endpoint_id",
        "columns": ["endpoint_id", "name", "region", "status", "category"],
    },
    "metrics": {
        "rows": 500_000,
        "foreign_key": "endpoint_id",
        "columns": ["endpoint_id", "metric_name", "value", "timestamp"],
    },
    "alerts": {
        "rows": 25_000,
        "primary_key": "alert_id",
        "foreign_key": "endpoint_id",
        "columns": ["alert_id", "endpoint_id", "severity", "message", "created_at"],
    },
}

print("Databricks host:", HOST)
print("Serverless SQL Warehouse:", SQL_WAREHOUSE_ID)
print("PostgreSQL database:", POSTGRES["database"])
print("Dataset context tables:", ", ".join(DATASET_CONTEXT.keys()))


## Setup — requests-based Unity Catalog API client

This client intentionally uses the Unity Catalog REST endpoints under:

- `/api/2.1/unity-catalog/...`
- `/api/2.1/lineage-tracking/...`

Authentication is expected through **`DATABRICKS_TOKEN`** in the environment.  
If the token is not available, the notebook stays executable and prints guidance instead of failing.


In [ ]:

@dataclass
class APIResult:
    ok: bool
    status_code: Optional[int]
    endpoint: str
    payload: Dict[str, Any]
    error: Optional[str] = None

class DatabricksUnityCatalogClient:
    def __init__(self, host: str, token: Optional[str], timeout: int = 30) -> None:
        self.host = host.rstrip("/")
        self.timeout = timeout
        self.session: Session = requests.Session()
        self.session.headers.update({"Content-Type": "application/json"})
        self.token = token
        if token:
            self.session.headers.update({"Authorization": f"Bearer {token}"})

    @property
    def enabled(self) -> bool:
        return bool(self.token)

    def _url(self, path: str) -> str:
        if not path.startswith("/"):
            path = "/" + path
        return f"{self.host}{path}"

    def _request(
        self,
        method: str,
        path: str,
        *,
        params: Optional[Dict[str, Any]] = None,
        json_body: Optional[Dict[str, Any]] = None,
    ) -> APIResult:
        endpoint = self._url(path)
        if not self.enabled:
            return APIResult(
                ok=False,
                status_code=None,
                endpoint=endpoint,
                payload={},
                error="DATABRICKS_TOKEN is not set. Export the token, then rerun this cell.",
            )

        try:
            response: Response = self.session.request(
                method=method.upper(),
                url=endpoint,
                params=params,
                json=json_body,
                timeout=self.timeout,
            )
            content_type = response.headers.get("Content-Type", "")
            if "application/json" in content_type:
                payload = response.json()
            else:
                payload = {"raw_text": response.text}

            if response.ok:
                return APIResult(
                    ok=True,
                    status_code=response.status_code,
                    endpoint=endpoint,
                    payload=payload,
                    error=None,
                )

            return APIResult(
                ok=False,
                status_code=response.status_code,
                endpoint=endpoint,
                payload=payload,
                error=payload.get("message") if isinstance(payload, dict) else response.text,
            )
        except requests.RequestException as exc:
            return APIResult(
                ok=False,
                status_code=None,
                endpoint=endpoint,
                payload={},
                error=f"{type(exc).__name__}: {exc}",
            )

    def list_catalogs(self) -> APIResult:
        return self._request("GET", "/api/2.1/unity-catalog/catalogs")

    def list_schemas(self, catalog_name: str) -> APIResult:
        return self._request(
            "GET",
            "/api/2.1/unity-catalog/schemas",
            params={"catalog_name": catalog_name},
        )

    def list_tables(self, catalog_name: str, schema_name: str) -> APIResult:
        return self._request(
            "GET",
            "/api/2.1/unity-catalog/tables",
            params={"catalog_name": catalog_name, "schema_name": schema_name},
        )

    def get_table(self, full_name: str) -> APIResult:
        return self._request(
            "GET",
            f"/api/2.1/unity-catalog/tables/{full_name}",
        )

    def get_table_lineage(
        self,
        *,
        table_name: str,
        include_entity_lineage: bool = True,
    ) -> APIResult:
        return self._request(
            "GET",
            "/api/2.1/lineage-tracking/table-lineage",
            params={
                "table_name": table_name,
                "include_entity_lineage": str(include_entity_lineage).lower(),
            },
        )

    def patch_table(self, table_name: str, json_body: Dict[str, Any]) -> APIResult:
        return self._request(
            "PATCH",
            f"/api/2.1/unity-catalog/tables/{table_name}",
            json_body=json_body,
        )

def print_api_result(result: APIResult, max_chars: int = 1200) -> None:
    print(f"Endpoint: {result.endpoint}")
    print(f"HTTP status: {result.status_code}")
    print(f"Success: {result.ok}")
    if result.error:
        print("Error:", result.error)
    rendered = json.dumps(result.payload, indent=2, default=str)
    if len(rendered) > max_chars:
        rendered = rendered[:max_chars] + "\n... [truncated]"
    print(rendered)

token = os.getenv("DATABRICKS_TOKEN")
client = DatabricksUnityCatalogClient(HOST, token=token)

print("Token available:", bool(token))
if not token:
    print("Set DATABRICKS_TOKEN in your shell or notebook environment to execute live Databricks API calls.")


## Catalog / Schema discovery

This section performs:

1. `GET /api/2.1/unity-catalog/catalogs`
2. `GET /api/2.1/unity-catalog/schemas?catalog_name=main`
3. `GET /api/2.1/unity-catalog/tables?...`
4. prints a small hierarchy tree

The code is defensive: if the `main` catalog or schemas are absent, it degrades cleanly.


In [ ]:

from collections import defaultdict

def choose_catalog_name(catalogs_payload: Dict[str, Any]) -> Optional[str]:
    catalogs = catalogs_payload.get("catalogs", [])
    names = [c.get("name") for c in catalogs if c.get("name")]
    if "main" in names:
        return "main"
    return names[0] if names else None

catalogs_result = client.list_catalogs()
print("=== Catalogs ===")
print_api_result(catalogs_result)

catalog_name = choose_catalog_name(catalogs_result.payload) if catalogs_result.ok else None
print("\nChosen catalog:", catalog_name)

schemas_result = client.list_schemas(catalog_name) if catalog_name else APIResult(
    ok=False,
    status_code=None,
    endpoint=f"{HOST}/api/2.1/unity-catalog/schemas",
    payload={},
    error="No catalog available to query.",
)

print("\n=== Schemas ===")
print_api_result(schemas_result)

schema_entries = schemas_result.payload.get("schemas", []) if schemas_result.ok else []
selected_schema = schema_entries[0]["name"] if schema_entries else None

tables_result = client.list_tables(catalog_name, selected_schema) if catalog_name and selected_schema else APIResult(
    ok=False,
    status_code=None,
    endpoint=f"{HOST}/api/2.1/unity-catalog/tables",
    payload={},
    error="No schema available to query.",
)

print("\n=== Tables ===")
print_api_result(tables_result)

def print_hierarchy_tree(
    catalog: Optional[str],
    schemas_payload: Dict[str, Any],
    tables_payload: Dict[str, Any],
) -> None:
    print("\n=== Hierarchy Tree ===")
    if not catalog:
        print("No catalog discovered.")
        return

    print(f"catalog: {catalog}")
    schemas = schemas_payload.get("schemas", []) or []
    tables = tables_payload.get("tables", []) or []

    tables_by_schema = defaultdict(list)
    for tbl in tables:
        schema_name = tbl.get("schema_name") or tbl.get("schema") or "unknown_schema"
        table_name = tbl.get("name") or tbl.get("table_name") or "unknown_table"
        tables_by_schema[schema_name].append(table_name)

    if not schemas:
        print("  └── no schemas returned")
        return

    for schema in schemas[:10]:
        schema_name = schema.get("name", "unknown_schema")
        print(f"  └── schema: {schema_name}")
        schema_tables = sorted(tables_by_schema.get(schema_name, []))
        if not schema_tables:
            print("      └── no tables returned")
            continue
        for table_name in schema_tables[:20]:
            print(f"      └── table: {table_name}")

print_hierarchy_tree(catalog_name, schemas_result.payload, tables_result.payload)


## Table metadata

This section retrieves one table definition and formats the useful metadata:

- columns
- owner
- created_at
- data_source_format

The lookup prefers the first discovered table. If no live table is available, the notebook exits the section cleanly.


In [ ]:

def fully_qualified_table_name(table_payload: Dict[str, Any]) -> Optional[str]:
    for key in ("full_name", "table_name", "name"):
        value = table_payload.get(key)
        if value:
            return value

    catalog = table_payload.get("catalog_name")
    schema = table_payload.get("schema_name")
    name = table_payload.get("name")
    if catalog and schema and name:
        return f"{catalog}.{schema}.{name}"
    return None

available_tables = tables_result.payload.get("tables", []) if tables_result.ok else []
sample_table_name = fully_qualified_table_name(available_tables[0]) if available_tables else None

table_result = client.get_table(sample_table_name) if sample_table_name else APIResult(
    ok=False,
    status_code=None,
    endpoint=f"{HOST}/api/2.1/unity-catalog/tables/<table-name>",
    payload={},
    error="No table discovered to describe.",
)

print("=== Table Metadata API Response ===")
print_api_result(table_result)

def format_columns(columns: Iterable[Dict[str, Any]]) -> str:
    rows = []
    header = f"{'column_name':<30} {'type_text':<20} {'nullable':<10} {'comment':<40}"
    rows.append(header)
    rows.append("-" * len(header))
    for column in columns:
        rows.append(
            f"{str(column.get('name', '')):<30} "
            f"{str(column.get('type_text', column.get('type_name', ''))):<20} "
            f"{str(column.get('nullable', '')):<10} "
            f"{str(column.get('comment', ''))[:40]:<40}"
        )
    return "\n".join(rows)

if table_result.ok:
    payload = table_result.payload
    print("\n=== Formatted Table Metadata ===")
    print("full_name        :", payload.get("full_name") or payload.get("name"))
    print("owner            :", payload.get("owner"))
    print("created_at       :", payload.get("created_at"))
    print("data_source_fmt  :", payload.get("data_source_format"))
    print("table_type       :", payload.get("table_type"))
    print("\nColumns")
    print(format_columns(payload.get("columns", [])))
else:
    print("\nTable metadata could not be retrieved live.")


## Lineage — conceptual and API shape

Unity Catalog lineage answers a key governance question:

> Which upstream columns, jobs, or queries contributed to this downstream table?

At a conceptual level, Databricks captures read and write events and resolves them into lineage edges.

Typical API shape shown in this notebook:

- `GET /api/2.1/lineage-tracking/table-lineage?table_name=<catalog.schema.table>`

That lets you inspect upstream and downstream relationships for a table.  
Column-level lineage is especially useful when downstream derived tables contain fields influenced by sensitive upstream columns.


In [ ]:

print("=== Lineage Concept ===")
print(
    textwrap.dedent(
        """
        Column-level lineage helps answer:
        - which source columns were read
        - which transformation produced the output
        - which downstream table/column now carries regulated meaning

        Citi telemetry example:
        - raw endpoint metrics land in a bronze table
        - curated SLA metrics are produced in silver
        - alerting or executive KPI tables are produced in gold

        If a regulated or sensitive attribute appears downstream, lineage explains the propagation path.
        """
    ).strip()
)

lineage_table_name = sample_table_name
print("\nLineage endpoint shape:")
print("GET /api/2.1/lineage-tracking/table-lineage?table_name=<catalog.schema.table>")

lineage_result = client.get_table_lineage(table_name=lineage_table_name) if lineage_table_name else APIResult(
    ok=False,
    status_code=None,
    endpoint=f"{HOST}/api/2.1/lineage-tracking/table-lineage",
    payload={},
    error="No table available for lineage lookup.",
)

print("\n=== Live Lineage Response ===")
print_api_result(lineage_result)


## Access control

Unity Catalog access control is hierarchical:

- **Catalog privileges** apply broadly
- **Schema privileges** narrow the scope
- **Table / View privileges** are most specific

Common pattern:

- grant usage on a catalog
- grant usage on a schema
- grant select or modify on specific tables

This differs from workspace-level ACLs:

- **workspace ACLs** govern notebooks, folders, jobs, dashboards, compute access
- **Unity Catalog ACLs** govern the **data asset itself**


In [ ]:

grant_sql = """
GRANT USAGE ON CATALOG main TO `data_engineers`;
GRANT USAGE ON SCHEMA main.telemetry TO `data_engineers`;
GRANT SELECT ON TABLE main.telemetry.endpoint_metrics TO `data_engineers`;

REVOKE MODIFY ON TABLE main.telemetry.endpoint_metrics FROM `analyst_contractors`;
"""

print("=== GRANT / REVOKE SQL Pattern ===")
print(grant_sql.strip())

print("\n=== UC vs Workspace ACLs ===")
comparison_rows = [
    ("Unity Catalog", "catalog/schema/table privileges", "who can read or modify governed data"),
    ("Workspace ACLs", "folders, notebooks, jobs, clusters, dashboards", "who can use workspace objects"),
]
header = f"{'control_plane':<18} {'scope':<45} {'question_answered'}"
print(header)
print("-" * len(header))
for row in comparison_rows:
    print(f"{row[0]:<18} {row[1]:<45} {row[2]}")


## Tags and classification

A practical governance pattern is applying tags or classifications to governed tables.

For Citi-like telemetry use cases, examples include:

- `classification=internal`
- `contains_pii=true`
- `regulatory_scope=sox`
- `domain=observability`

The example below shows the **PATCH** structure for a table update call.  
It is guarded by `RUN_TAG_PATCH = False` so the notebook remains safe to run top-to-bottom.


In [ ]:

RUN_TAG_PATCH = False

example_tags_payload = {
    "properties": {
        "classification": "internal",
        "contains_pii": "false",
        "domain": "observability",
        "business_owner": "telemetry_platform",
    }
}

print("=== PATCH Pattern ===")
print(f"PATCH /api/2.1/unity-catalog/tables/{{table_name}}")
print(json.dumps(example_tags_payload, indent=2))

print("\n=== Citi PII Classification Example ===")
print(
    textwrap.dedent(
        """
        Citi telemetry usually centers on endpoint_id, latency, throughput, and error metrics.
        Those fields are operational and often not directly PII.
        But if a downstream enrichment introduces owner_email, oncall_contact, or user-linked metadata,
        the table should be tagged and controlled accordingly.
        """
    ).strip()
)

patch_result = APIResult(
    ok=False,
    status_code=None,
    endpoint=f"{HOST}/api/2.1/unity-catalog/tables/<table-name>",
    payload={},
    error="PATCH not executed. Set RUN_TAG_PATCH=True only when you intend to modify table metadata.",
)

if RUN_TAG_PATCH and sample_table_name:
    patch_result = client.patch_table(sample_table_name, example_tags_payload)

print("\n=== PATCH Execution Result ===")
print_api_result(patch_result)


## What just happened

Unity Catalog is Databricks' answer to centralized data governance:

- one governance plane across catalogs, schemas, and tables
- API-driven discovery of metadata
- lineage for impact analysis and regulatory traceability
- privilege inheritance and precise data access control
- table properties and tags for classification

For the Citi telemetry narrative, this maps directly to enterprise governance needs:  
**sensitive fields require classification, lineage visibility, and policy-controlled access at the data layer**, not just the workspace layer. fileciteturn1file0


In [ ]:

summary = {
    "title": "Unity Catalog — Governance, Lineage, Access Control on Databricks",
    "host": HOST,
    "warehouse_id": SQL_WAREHOUSE_ID,
    "catalog_discovery_attempted": True,
    "sample_catalog": catalog_name,
    "sample_schema": selected_schema,
    "sample_table": sample_table_name,
    "lineage_attempted": bool(lineage_table_name),
    "tag_patch_executed": RUN_TAG_PATCH,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
}

print(json.dumps(summary, indent=2))
